# Vortex Colab GPU capability check (environment probe only)

No model download or inference is performed. Pin dependencies before a real experiment. Run with an existing Python 3.11+ kernel; PyTorch is optional. This notebook is self-contained and needs no repository checkout or installation cell.

Use only an available free environment. GPU selection is a manual provider setting, not an action performed by this notebook. If a free GPU is unavailable, keep the CPU environment or defer; do not provision paid resources.

In Colab, inspect the current runtime/hardware setting and use an available free GPU only if authorized. This probe also works with a CPU runtime; it does not authenticate, mount Drive or modify runtime settings.


In [ ]:
from contextlib import redirect_stderr, redirect_stdout
import importlib
import importlib.util
import io
import json
import platform
import shutil
import sys
from typing import Literal, TypedDict


class Device(TypedDict):
    index: int
    name: str
    total_bytes: int
    compute_capability: list[int]


class ProbeError(TypedDict):
    stage: str
    type: str


class Capability(TypedDict):
    platform: str
    python: str
    status: Literal["torch_unavailable", "cuda_unavailable", "cuda_available", "probe_error"]
    torch_installed: bool | None
    torch_available: bool
    torch_version: str | None
    torch_cuda_version: str | None
    nvidia_smi_available: bool | None
    cuda_available: bool | None
    device_count: int | None
    devices: list[Device]
    probe_error: ProbeError | None


def capability() -> Capability:
    """Return visible CUDA metadata, with null for facts that were not established.

    torch_available means import succeeded; nvidia_smi_available is PATH presence
    only. Device memory is total capacity in bytes, not free memory or a workload
    guarantee. Optional-library errors expose stage/type, never raw messages.
    """
    out: Capability = {
        "platform": platform.system(), "python": sys.version.split()[0],
        "status": "torch_unavailable", "torch_installed": None,
        "torch_available": False, "torch_version": None, "torch_cuda_version": None,
        "nvidia_smi_available": None, "cuda_available": None, "device_count": None,
        "devices": [], "probe_error": None,
    }
    stage = "nvidia_smi_lookup"
    try:
        out["nvidia_smi_available"] = shutil.which("nvidia-smi") is not None
        stage = "torch_lookup"
        out["torch_installed"] = importlib.util.find_spec("torch") is not None
        if not out["torch_installed"]:
            return out
        stage = "torch_import"
        torch = importlib.import_module("torch")
        out["torch_available"] = True
        version, cuda_version = torch.__version__, torch.version.cuda
        if not isinstance(version, str) or (cuda_version is not None and not isinstance(cuda_version, str)):
            raise TypeError("Unexpected version metadata")
        out["torch_version"], out["torch_cuda_version"] = version, cuda_version
        stage = "cuda_availability"
        available = torch.cuda.is_available()
        if type(available) is not bool:
            raise TypeError("Unexpected CUDA availability")
        out["cuda_available"] = available
        if not available:
            out["status"], out["device_count"] = "cuda_unavailable", 0
            return out
        stage = "device_count"
        count = torch.cuda.device_count()
        if type(count) is not int or count < 1:
            raise ValueError("Inconsistent CUDA device count")
        stage = "device_properties"
        devices: list[Device] = []
        for index in range(count):
            props = torch.cuda.get_device_properties(index)
            name, memory, major, minor = props.name, props.total_memory, props.major, props.minor
            if not isinstance(name, str) or not name:
                raise ValueError("Invalid device name")
            if type(memory) is not int or memory < 1:
                raise ValueError("Invalid total device memory")
            if type(major) is not int or major < 1 or type(minor) is not int or minor < 0:
                raise ValueError("Invalid compute capability")
            devices.append({"index": index, "name": name, "total_bytes": memory,
                            "compute_capability": [major, minor]})
        # Publish an inventory only after every visible device was queried successfully.
        out["devices"], out["device_count"], out["status"] = devices, count, "cuda_available"
    except Exception as error:
        out["status"] = "probe_error"
        out["probe_error"] = {"stage": stage, "type": type(error).__name__}
    return out


In [ ]:
def main() -> int:
    """Print one JSON report; exit 0 for normal CPU/CUDA outcomes, 2 for probe errors."""
    previous = sys.dont_write_bytecode
    try:
        sys.dont_write_bytecode = True
        # Python-level chatter is withheld; native diagnostics may bypass these streams.
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            report = capability()
    finally:
        sys.dont_write_bytecode = previous
    print(json.dumps(report, indent=2, allow_nan=False))
    return 2 if report["status"] == "probe_error" else 0

_probe_exit_code = main()


## Interpreting the JSON

- Python version and operating-system family describe the active kernel, not a pinned benchmark environment.
- PyTorch version and its CUDA build version are separate. A null CUDA build does not establish whether a standalone toolkit or driver is installed.
- Status is one of: torch_unavailable (package absent), cuda_unavailable (PyTorch imports but CUDA is unavailable), cuda_available (all visible device properties queried), or probe_error (metadata could not be established).
- devices lists every GPU visible to this PyTorch CUDA process: logical index, name, total_bytes and compute_capability. Total memory is capacity, not currently free memory or a model-fit guarantee. Visibility is not an exhaustive physical GPU inventory; a CPU PyTorch build cannot enumerate CUDA GPUs.
- nvidia_smi_available is only executable presence on PATH. The tool is never invoked, and presence does not prove a usable driver or GPU.
- Null means unknown/not queried; zero devices is reported only after CUDA is confirmed unavailable. On a query failure, no partial inventory is published.
- probe_error includes only stage and exception type. Check the package/kernel/runtime configuration at that stage; raw exception messages and Python-level dependency stdout/stderr are withheld because they can contain local paths or sensitive information. Native library diagnostics can bypass these Python streams and may still appear in cell output, so keep all output private and clear it before publication. No automatic repair or install is attempted.

The probe performs metadata queries only: no tensors, model loading, shell commands, network requests, provisioning or performance workload. A GPU property query can initialize a CUDA context; this is not an inference or memory-capacity test. It records no account/session IDs, hostnames, environment variables or credentials.

Run both code cells in order. The probe prints a JSON report; native diagnostics may also appear. A query error does not terminate the notebook kernel. Retain any output privately and verify exports before a temporary session ends, as described in [reproducibility](../../docs/reproducibility.md). Before publishing the notebook, clear every code output and execution count and remove attachments. Follow the [results boundary](../../results/README.md); a successful probe is not benchmark evidence or permission for Phase 0.
